In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
PROJECT_ID='kolkata-uhi'

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print('Eath Engine Initialized')

In [ ]:
CENTER_LON = 74.25
CENTER_LAT = 21.37

centre = ee.Geometry.Point([CENTER_LON, CENTER_LAT])
aoi = centre.buffer(20_000).bounds()
sample_point = centre

In [ ]:
START_DATE='2025-01-01'
END_DATE='2025-12-31'
MAX_CLOUD=60

raw_collection=(
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD))
)

scene_count=raw_collection.size().getInfo()
print('Number of Matching scenes:', scene_count)

if scene_count==0:
  raise ValueError("No scenees found")

In [ ]:
def preprocess_landsat8(image):
  qa = image.select('QA_PIXEL')

  clear_mask=(
      qa.bitwiseAnd(1<<1).eq(0)
      .And(qa.bitwiseAnd(1<<2).eq(0))
      .And(qa.bitwiseAnd(1<<3).eq(0))
      .And(qa.bitwiseAnd(1<<4).eq(0))
      .And(qa.bitwiseAnd(1<<5).eq(0))
  )
  saturation_mask = image.select('QA_RADSAT').eq(0)
  optical = image.select('SR_B[1-7]').multiply(0.0000275).add(-0.2)

  return(
    image.addBands(optical, None, True)
    .updateMask(clear_mask)
    .updateMask(saturation_mask)
  )

raw_image = ee.Image(raw_collection.sort('CLOUD_COVER').first())
image=preprocess_landsat8(raw_image)

print('Product ID:', raw_image.get('LANDSAT_PRODUCT_ID').getInfo())

In [ ]:
print(raw_image.toDictionary(['LANDSAT_PRODUCT_ID','DATE_ACQUIRED','CLOUD_COVER','WRS_PATH','WRS_ROW']).getInfo())
  # r for the Table 4 caption

In [ ]:
Map=geemap.Map()
Map.centerObject(aoi, 10)

true_color={
    'bands':['SR_B4', 'SR_B3', 'SR_B2'],
    'min':0.0,
    'max':0.30
}

false_color={
    'bands':['SR_B5', 'SR_B4', 'SR_B3'],
    'min':0.0,
    'max':0.35
}

Map.addLayer(image, true_color, 'True Color')
Map.addLayer(image, false_color, 'false Color', False)
Map.addLayer(aoi, {'color':'yellow'}, 'AOI', False)
Map.addLayer(sample_point, {'color':'red'}, 'sample point')
Map

In [ ]:
samples = {
    'Vegetation': ee.Geometry.Point([73.77356019002816, 24.580339427513646]),       # Hills/Forest area
    'Water': ee.Geometry.Point([73.81613221151254, 24.569099346456674]),            # Bay / Water body
    'Bare Soil / Built-up': ee.Geometry.Point([73.65065064412973,  24.51647612514387])  # Urban/Panvel center
}

bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
wavelength_um = [0.443, 0.482, 0.562, 0.655, 0.865, 1.609, 2.201]

# 2. Add sample points to your geemap layout for visual verification
for surface_name, pt in samples.items():
    Map.addLayer(pt, {}, f'Sample: {surface_name}')

# 3. Loop through each point and extract spectral reflectance
spectral_list = []

for surface_name, pt in samples.items():
    sample_area = pt.buffer(45)

    spectrum = (
        image.select(bands)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=sample_area,
            scale=30,
            maxPixels=1_000_000,
        )
        .getInfo()
    )

    # Get reflectance values for each band
    reflectance = [spectrum.get(b, np.nan) for b in bands]

    # Store in list
    for w, r in zip(wavelength_um, reflectance):
        spectral_list.append({
            'Surface_Type': surface_name,
            'Wavelength_um': w,
            'Surface_reflectance': r
        })

# 4. Create master DataFrame
spectral_df = pd.DataFrame(spectral_list)
print("Extracted Spectral Data:")
display(spectral_df.head(21))

# 5. Plot all 3 signatures on a single graph
plt.figure(figsize=(9, 5))

for surface_name, group in spectral_df.groupby('Surface_Type'):
    plt.plot(
        group['Wavelength_um'],
        group['Surface_reflectance'],
        marker='o',
        linewidth=2,
        label=surface_name
    )

plt.xlabel('Wavelength (micrometres)', fontsize=12)
plt.ylabel('Surface Reflectance', fontsize=12)
plt.title('Comparison of Spectral Signatures (Landsat 8 L2)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Surface Type', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
spectral_df

In [ ]:
#visualizing each spectral bands
band_names = [
    ('SR_B1', 'Coastal aerosol'),
    ('SR_B2', 'Blue'),
    ('SR_B3', 'Green'),
    ('SR_B4', 'Red'),
    ('SR_B5', 'NIR'),
    ('SR_B6', 'SWIR1'),
    ('SR_B7', 'SWIR2'),
]
aoI = centre.buffer(60_000).bounds()
BandMap = geemap.Map()
BandMap.centerObject(aoi, 10)

for band, label in band_names:
    BandMap.addLayer(
        image.select(band).clip(aoI),
        {'min': 0.0, 'max': 0.35},
        label,
        False,
    )

BandMap.addLayer(
    image.select('SR_B4').clip(aoI),
    {'min': 0.0, 'max': 0.35},
    'Red - active'
)

BandMap.addLayer(sample_point, {'color': 'red'}, 'Sample point')

BandMap

# Implementing NDVI, NDWI, SAVI, NBR

In [ ]:
#select the reflectance bands once for clarity
green = image.select("SR_B3")
red = image.select("SR_B4")
nir = image.select("SR_B5")
swir2 = image.select("SR_B7")

#NDVI
ndvi = image.expression(
    "(NIR - RED)/(NIR + RED)",
    {"NIR":nir, "RED":red}
).rename("NDVI")

#NDWI
ndwi = image.expression(
  "(GREEN - NIR)/(GREEN + NIR)",
  {"GREEN": green, "NIR": nir}
).rename("NDWI")

#SAVI
L = 0.5
savi = image.expression(
    "((NIR - RED)/(NIR + RED + L))*(1 + L)",
    {"NIR":nir, "RED":red, "L":L}
).rename("SAVI")

#NBR
nbr = image.expression(
    "(NIR - SWIR2)/(NIR + SWIR2)",
    {"NIR":nir, "SWIR2":swir2}
).rename("NBR")

index_image = image.addBands([ndvi, ndwi, savi, nbr])
print("Index bands added:")
print(index_image.bandNames().getInfo())

In [ ]:
# Visualizing the Maps
Map = geemap.Map()
Map.centerObject(aoi, 10)

# true_color = {
#     "bands":['SR_B4', 'SR_B3', 'SR_B2'],
#     "min": 0.0,
#     "max": 0.60
# }

# false_color = {
#     "bands":['SR_B5', 'SR_B4', 'SR_B3'],
#     "min": 0.0,
#     "max": 0.30
# }

# Map.addLayer(image.clip(aoi), true_color, "True Color Composite", True)
# Map.addLayer(image.clip(aoi), false_color, "False Color Composite", False)

# Map.addLayer(
#     ndvi.clip(aoi),
#     {"min":0.25, "max":0.72, "palette":['#faffec', '#e2ff8e', '#a8c557']},
#     "NDVI",
#     False,
# )

# Map.addLayer(
#     ndwi.clip(aoi),
#     {"min":-1, "max":1, "palette":['a6611a', 'f5f5f5', '018571']},
#     "NDWI",
#     False,
# )

# Map.addLayer(
#     savi.clip(aoi),
#     {"min":-1, "max":1, "palette":['8c510a', 'f6e8c3', '01655e']},
#     "SAVI",
#     False,
# )

Map.addLayer(
    nbr.clip(aoi),
    {"min":-1, "max":1, "palette":['b2182b', 'f7f7f7', '2166ac']},
    "NBR",
    False,
)

Map.addLayer(aoi, {"color":"black"}, "AOI", False)
Map

# Quantitative Evaluation

In [ ]:
# Part G — Quantitative evaluation

indices = index_image.select(["NDVI", "NDWI", "SAVI", "NBR"])

stats = indices.reduceRegion(
    reducer=(
        ee.Reducer.mean()
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
        .combine(ee.Reducer.minMax(), sharedInputs=True)
    ),
    geometry=aoi,
    scale=30,
    maxPixels=1e8,
    bestEffort=True,
)

stats_dict = stats.getInfo()

summary_rows = []
for name in ["NDVI", "NDWI", "SAVI", "NBR"]:
    summary_rows.append({
        "Index": name,
        "Mean": stats_dict.get(f"{name}_mean"),
        "StdDev": stats_dict.get(f"{name}_stdDev"),
        "Min": stats_dict.get(f"{name}_min"),
        "Max": stats_dict.get(f"{name}_max"),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

# Range Check

In [ ]:
sample_fc = indices.sample(
    region=aoi,
    scale=30,
    numPixels=2500,
    seed=42,
    geometries=False,
    tileScale=2,
)

sample_info = sample_fc.getInfo()

sample_df = pd.DataFrame(
    [feature["properties"] for feature in sample_info["features"]]
).dropna()

print("Number of sampled clear pixels:", len(sample_df))
sample_df.head()

In [ ]:
# Plot a separate histogram for each index.
for name in ["NDVI", "NDWI", "SAVI", "NBR"]:
    plt.figure(figsize=(7, 4))
    plt.hist(sample_df[name], bins=40)
    plt.xlabel(name)
    plt.ylabel("Number of sampled pixels")
    plt.title(f"Distribution of {name}")
    plt.grid(alpha=0.2)
    plt.show()

In [ ]:
sample_df["NDVI_minus_SAVI"] = sample_df["NDVI"] - sample_df["SAVI"]

comparison = sample_df[["NDVI", "SAVI", "NDVI_minus_SAVI"]].describe()
comparison

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(sample_df["NDVI"], sample_df["SAVI"], s=8, alpha=0.35)

plt.xlabel("NDVI")
plt.ylabel("SAVI")
plt.title("NDVI versus SAVI")
plt.axline((0, 0), slope=1, linestyle="--")
plt.grid(alpha=0.2)
plt.show()

# Build Simple Interpretation marks

In [ ]:
vegetation_candidate = ndvi.gt(0.40).rename("Vegetation Candidate")
water_candidate = ndwi.gt(0.00).rename("Water_Candidate")

MaskMap = geemap.Map()
MaskMap.centerObject(aoi, 10)
MaskMap.addLayer(image.clip(aoi), true_color, "True Color", True)
# MaskMap.addLayer(
#     vegetation_candidate.selfMask().clip(aoi),
#     {"palette":['00AA00']},
#     "NDVI > 0.40",
#     False,
# )
MaskMap.addLayer(
    water_candidate.selfMask().clip(aoi),
    {"palette":['00AA00']},
    "NDWI > 0",
    False,
)
MaskMap

# Export the index stack

In [ ]:
export_indices = index_image.select(
    ['NDVI', 'NDWI', 'SAVI', 'NBR']
).clip(aoi)

task = ee.batch.Export.image.toDrive(
    image=export_indices,
    description="Landsat8_Spectral_Indices",
    folder="GEE_exports",
    fileNamePrefix="Landsat8_NDVI_NDWI_SAVI_NBR",
    region=aoi,
    scale=30,
    maxPixels=1e9,
    fileFormat="GeoTIFF"
)

task.start()

print("Export prepared. Uncomment task.start() to submit it")